# 夫琅禾费衍射

### 串行程序

In [6]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# Basic utilities
# ============================================================

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def savefig_no_margin(fig, filename, dpi=450):
    fig.savefig(
        filename,
        dpi=dpi,
        bbox_inches="tight",
        pad_inches=0.0,
    )


def set_plot_style(plot_params):
    plt.rcParams["font.family"] = plot_params["FONT_FAMILY"]
    plt.rcParams["mathtext.fontset"] = plot_params["MATH_FONTSET"]
    plt.rcParams["axes.linewidth"] = plot_params["AXES_LINEWIDTH"]
    plt.rcParams["xtick.direction"] = "in"
    plt.rcParams["ytick.direction"] = "in"
    plt.rcParams["xtick.top"] = True
    plt.rcParams["ytick.right"] = True
    plt.rcParams["figure.dpi"] = plot_params["FIGURE_DPI"]


def beautify_axes(ax, xlabel=None, ylabel=None, plot_params=None):
    if plot_params is None:
        label_size = 15
        tick_size = 12
        axes_lw = 1.2
        tick_lw = 1.2
        tick_len = 5
    else:
        label_size = plot_params["LABEL_SIZE"]
        tick_size = plot_params["TICK_SIZE"]
        axes_lw = plot_params["AXES_LINEWIDTH"]
        tick_lw = plot_params["TICK_WIDTH"]
        tick_len = plot_params["TICK_LENGTH"]

    if xlabel is not None:
        ax.set_xlabel(xlabel, fontsize=label_size)
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontsize=label_size)

    ax.tick_params(labelsize=tick_size, width=tick_lw, length=tick_len)

    for spine in ax.spines.values():
        spine.set_linewidth(axes_lw)


def at_most_three_ticks(values):
    """Return a tick list with at most three entries."""
    values = list(values)
    if len(values) <= 3:
        return values
    return [values[0], values[len(values) // 2], values[-1]]


def three_ticks_from_range(vmin, vmax):
    """Use only left, center, and right ticks."""
    return [vmin, 0.5 * (vmin + vmax), vmax]


def set_ticks_from_params(ax, xticks=None, yticks=None):
    if xticks is not None:
        ax.set_xticks(at_most_three_ticks(xticks))
    if yticks is not None:
        ax.set_yticks(at_most_three_ticks(yticks))


# ============================================================
# Geometry helpers
# ============================================================

def lx_total(params):
    return (
        params["LX_S_LEFT"]
        + params["LX_N"]
        + params["LX_S_RIGHT"]
    )


def nsites(params):
    return lx_total(params) * params["WY"]


def site_index(x, y, params):
    return x * params["WY"] + y


def region_of_x(x, params):
    if x < params["LX_S_LEFT"]:
        return "L"
    if x < params["LX_S_LEFT"] + params["LX_N"]:
        return "N"
    return "R"


def onsite_mu(x, params):
    if region_of_x(x, params) == "N":
        return params["MU_N"]
    return params["MU_S"]


def onsite_potential(x, params):
    if region_of_x(x, params) == "N":
        return params["U_N"]
    return 0.0


def is_weak_link_x_bond(x, params):
    """
    x-bond connects x -> x+1.

    The orbital phase is applied to the weak-link region, including both
    S-N interface bonds:
        x = LX_S_LEFT - 1, ..., LX_S_LEFT + LX_N - 1.
    """
    return (
        params["LX_S_LEFT"] - 1
        <= x
        <= params["LX_S_LEFT"] + params["LX_N"] - 1
    )


def number_of_flux_bonds(params):
    return params["LX_N"] + 1


# ============================================================
# Peierls phase
# ============================================================

def peierls_phase_x(y, flux_phi0, params):
    """
    Electron Peierls phase for one x-direction hopping at transverse position y.

    Gauge:
        A_x = -B (y - y0)

    flux_phi0 = Phi / Phi0, where Phi0 = h/(2e).

    Electron flux quantum is h/e = 2 Phi0, so the electron Peierls phase per
    plaquette is pi times the flux per plaquette measured in Phi0 units.

    The phase is distributed uniformly over the active x-bonds across the weak
    link. Then the Cooper-pair phase variation across the width is
    approximately 2*pi*Phi/Phi0.
    """
    wy = params["WY"]
    if wy <= 1:
        return 0.0

    y0 = 0.5 * (wy - 1)
    nxb = number_of_flux_bonds(params)

    flux_per_cell_phi0 = flux_phi0 / (nxb * (wy - 1))

    theta = -np.pi * flux_per_cell_phi0 * (y - y0)

    return theta


# ============================================================
# Hamiltonian construction
# ============================================================

def build_electron_hamiltonian(flux_phi0, params):
    """
    Electron Hamiltonian H_e for a 2D finite-width SNS junction.

    The hole block is generated in BdG as -H_e^*.
    """
    ns = nsites(params)
    lx = lx_total(params)
    wy = params["WY"]

    t = params["T_HOP"]

    H = np.zeros((ns, ns), dtype=complex)

    # Onsite terms
    for x in range(lx):
        for y in range(wy):
            i = site_index(x, y, params)

            mu = onsite_mu(x, params)
            U = onsite_potential(x, params)

            H[i, i] = -mu + U

    # Hopping terms
    for x in range(lx):
        for y in range(wy):
            i = site_index(x, y, params)

            # x hopping: (x, y) -> (x+1, y)
            if x + 1 < lx:
                j = site_index(x + 1, y, params)

                theta = 0.0
                if is_weak_link_x_bond(x, params):
                    theta = peierls_phase_x(y, flux_phi0, params)

                hop = -t * np.exp(1j * theta)

                H[i, j] = hop
                H[j, i] = np.conjugate(hop)

            # y hopping: (x, y) -> (x, y+1)
            if y + 1 < wy:
                j = site_index(x, y + 1, params)

                hop = -t

                H[i, j] = hop
                H[j, i] = np.conjugate(hop)

    # Optional onsite interface barrier on the first and last normal layers
    if abs(params["U_INTERFACE"]) > 1.0e-14:
        x_left_n = params["LX_S_LEFT"]
        x_right_n = params["LX_S_LEFT"] + params["LX_N"] - 1

        for y in range(wy):
            for x in [x_left_n, x_right_n]:
                i = site_index(x, y, params)
                H[i, i] += params["U_INTERFACE"]

    return H


def build_pairing_matrix(phi, params):
    """
    Onsite s-wave pairing.

    Left S:
        Delta_L = Delta0 exp(-i phi/2)

    Right S:
        Delta_R = Delta0 exp(+i phi/2)

    Normal region:
        Delta = 0
    """
    ns = nsites(params)
    lx = lx_total(params)
    wy = params["WY"]

    delta0 = params["DELTA0"]

    D = np.zeros((ns, ns), dtype=complex)

    phase_L = np.exp(-0.5j * phi)
    phase_R = np.exp(+0.5j * phi)

    for x in range(lx):
        reg = region_of_x(x, params)

        if reg == "L":
            delta = delta0 * phase_L
        elif reg == "R":
            delta = delta0 * phase_R
        else:
            delta = 0.0

        if abs(delta) > 0:
            for y in range(wy):
                i = site_index(x, y, params)
                D[i, i] = delta

    return D


def build_bdg_from_he_and_pairing(He, D):
    """
    BdG Hamiltonian in basis:
        (all electrons, all holes)

    H_BdG =
        [ H_e          Delta     ]
        [ Delta^dag   -H_e^*    ]
    """
    ns = He.shape[0]
    dim = 2 * ns

    Hbdg = np.zeros((dim, dim), dtype=complex)

    Hbdg[:ns, :ns] = He
    Hbdg[:ns, ns:] = D
    Hbdg[ns:, :ns] = D.conjugate().T
    Hbdg[ns:, ns:] = -He.conjugate()

    return Hbdg


# ============================================================
# Josephson energy and current
# ============================================================

def bdg_ground_state_energy_from_he(He, phi, params):
    """
    Zero-temperature BdG ground-state energy.

    Up to phi-independent constants:
        F(phi) = -1/2 sum_{E_n > 0} E_n

    The spin degeneracy is applied later to the current.
    """
    D = build_pairing_matrix(phi, params)
    Hbdg = build_bdg_from_he_and_pairing(He, D)

    evals = np.linalg.eigvalsh(Hbdg)
    positive = evals[evals > params["EIG_CUTOFF"]]

    F = -0.5 * np.sum(positive)

    return float(np.real(F))


def periodic_derivative(values, dx):
    """
    Periodic central derivative for arrays sampled with endpoint=False.
    """
    return (np.roll(values, -1) - np.roll(values, 1)) / (2.0 * dx)


def calculate_cpr_for_flux(flux_phi0, phi_values, params):
    """
    Calculate F(phi) and I(phi) at fixed flux.

    Josephson current:
        I = (2e/hbar) dF/dphi.

    Here e=hbar=1 units are used. The final current is multiplied by
    SPIN_DEGENERACY.
    """
    He = build_electron_hamiltonian(flux_phi0, params)

    F_values = np.zeros_like(phi_values, dtype=float)

    for ip, phi in enumerate(phi_values):
        F_values[ip] = bdg_ground_state_energy_from_he(He, phi, params)

    dphi = phi_values[1] - phi_values[0]

    I_one_block = (
        params["CURRENT_SIGN"]
        * 2.0
        * periodic_derivative(F_values, dphi)
    )

    I_total = params["SPIN_DEGENERACY"] * I_one_block

    if params["SUBTRACT_CURRENT_MEAN"]:
        I_total = I_total - np.mean(I_total)

    return F_values, I_total


# ============================================================
# Calculation and saving
# ============================================================

def make_grids(params):
    flux_values = np.linspace(
        params["FLUX_MIN"],
        params["FLUX_MAX"],
        params["N_FLUX"],
    )

    phi_values = np.linspace(
        params["PHI_MIN"],
        params["PHI_MAX"],
        params["N_PHI"],
        endpoint=False,
    )

    return flux_values, phi_values


def calculate_and_save_data(params):
    """
    Main calculation function.

    This function only calculates the data and saves it to DATA_FILE.
    It does not make plots.
    """
    ensure_dir(params["OUTDIR"])

    flux_values, phi_values = make_grids(params)

    n_flux = len(flux_values)
    n_phi = len(phi_values)

    F_map = np.zeros((n_flux, n_phi), dtype=float)
    I_map = np.zeros((n_flux, n_phi), dtype=float)

    Ic_abs = np.zeros(n_flux, dtype=float)
    Ic_plus = np.zeros(n_flux, dtype=float)
    Ic_minus = np.zeros(n_flux, dtype=float)

    for iF, flux in enumerate(flux_values):
        print(
            f"Flux {iF + 1:4d}/{n_flux}: "
            f"Phi/Phi0 = {flux: .6f}"
        )

        F_values, I_values = calculate_cpr_for_flux(
            flux_phi0=flux,
            phi_values=phi_values,
            params=params,
        )

        F_map[iF, :] = F_values
        I_map[iF, :] = I_values

        Ic_plus[iF] = np.max(I_values)
        Ic_minus[iF] = np.min(I_values)
        Ic_abs[iF] = np.max(np.abs(I_values))

    params_json = json.dumps(params, indent=2, sort_keys=True)

    np.savez_compressed(
        params["DATA_FILE"],
        flux_values=flux_values,
        phi_values=phi_values,
        F_map=F_map,
        I_map=I_map,
        Ic_abs=Ic_abs,
        Ic_plus=Ic_plus,
        Ic_minus=Ic_minus,
        params_json=np.array(params_json),
    )

    print("=" * 80)
    print(f"Saved data to: {params['DATA_FILE']}")
    print("=" * 80)

    return params["DATA_FILE"]


# ============================================================
# Loading saved data
# ============================================================

def load_saved_data(data_file):
    loaded = np.load(data_file, allow_pickle=True)

    data = {key: loaded[key] for key in loaded.files}

    if "params_json" in data:
        params_json = data["params_json"].item()
        params = json.loads(params_json)
    else:
        params = {}

    return data, params


# ============================================================
# Plot helpers
# ============================================================

def analytic_fraunhofer(flux_values):
    """
    Standard Fraunhofer envelope:
        |sin(pi f)/(pi f)|

    where f = Phi/Phi0.
    """
    x = np.pi * flux_values
    y = np.ones_like(x)

    mask = np.abs(x) > 1.0e-12
    y[mask] = np.abs(np.sin(x[mask]) / x[mask])

    return y


def nearest_index(array, value):
    return int(np.argmin(np.abs(array - value)))


def normalize_ic(flux_values, Ic_abs, plot_params):
    mode = plot_params["IC_NORMALIZATION"]

    if mode == "max":
        return Ic_abs / (np.max(Ic_abs) + 1.0e-30)

    if mode == "zero_flux":
        idx0 = nearest_index(flux_values, 0.0)
        return Ic_abs / (Ic_abs[idx0] + 1.0e-30)

    raise ValueError("IC_NORMALIZATION must be 'max' or 'zero_flux'.")


# ============================================================
# Plot: geometry preview
# ============================================================

def plot_geometry_preview(params, plot_params):
    ensure_dir(plot_params["PLOT_DIR"])

    lx = lx_total(params)
    wy = params["WY"]

    region_map = np.zeros((wy, lx))

    for x in range(lx):
        reg = region_of_x(x, params)

        if reg == "L":
            val = 0.0
        elif reg == "N":
            val = 0.5
        else:
            val = 1.0

        region_map[:, x] = val

    fig, ax = plt.subplots(figsize=plot_params["FIGSIZE_GEOMETRY"])

    ax.imshow(
        region_map,
        origin="lower",
        aspect="auto",
        cmap=plot_params["GEOMETRY_CMAP"],
        interpolation="nearest",
        extent=[0, lx, 0, wy],
    )

    ax.axvline(params["LX_S_LEFT"], color="w", lw=1.4, ls="--")
    ax.axvline(params["LX_S_LEFT"] + params["LX_N"], color="w", lw=1.4, ls="--")

    ax.text(
        0.5 * params["LX_S_LEFT"],
        0.5 * wy,
        "S",
        color="w",
        ha="center",
        va="center",
        fontsize=16,
    )

    ax.text(
        params["LX_S_LEFT"] + 0.5 * params["LX_N"],
        0.5 * wy,
        "N",
        color="w",
        ha="center",
        va="center",
        fontsize=16,
    )

    ax.text(
        params["LX_S_LEFT"] + params["LX_N"] + 0.5 * params["LX_S_RIGHT"],
        0.5 * wy,
        "S",
        color="w",
        ha="center",
        va="center",
        fontsize=16,
    )

    beautify_axes(
        ax,
        r"$x$",
        r"$y$",
        plot_params,
    )

    geometry_xticks = plot_params.get("GEOMETRY_XTICKS", None)
    if geometry_xticks is None:
        geometry_xticks = three_ticks_from_range(0, lx)

    geometry_yticks = plot_params.get("GEOMETRY_YTICKS", None)
    if geometry_yticks is None:
        geometry_yticks = three_ticks_from_range(0, wy)

    set_ticks_from_params(ax, geometry_xticks, geometry_yticks)

    savefig_no_margin(
        fig,
        os.path.join(plot_params["PLOT_DIR"], "geometry_preview.png"),
        dpi=plot_params["SAVE_DPI"],
    )

    plt.close(fig)


# ============================================================
# Plot: Fraunhofer pattern
# ============================================================

def plot_fraunhofer(data, params, plot_params):
    ensure_dir(plot_params["PLOT_DIR"])

    flux = data["flux_values"]
    Ic = data["Ic_abs"]

    Ic_norm = normalize_ic(flux, Ic, plot_params)

    fig, ax = plt.subplots(figsize=plot_params["FIGSIZE_FRAUNHOFER"])

    ax.plot(
        flux,
        Ic_norm,
        color=plot_params["COLOR_IC"],
        lw=plot_params["LW_MAIN"],
        label=plot_params["LABEL_NUMERICAL"],
    )

    if plot_params["SHOW_ANALYTIC_FRAUNHOFER"]:
        ax.plot(
            flux,
            analytic_fraunhofer(flux),
            color=plot_params["COLOR_ANALYTIC"],
            lw=plot_params["LW_ANALYTIC"],
            ls="--",
            label=r"$|\sin(\pi\Phi/\Phi_0)/(\pi\Phi/\Phi_0)|$",
        )

    ax.axhline(0.0, color=plot_params["COLOR_GUIDE"], lw=0.8, ls="--", zorder=0)
    ax.axvline(0.0, color=plot_params["COLOR_GUIDE"], lw=0.8, ls="--", zorder=0)

    ax.set_xlim(plot_params["FLUX_XLIM"])
    ax.set_ylim(plot_params["IC_YLIM"])

    ax.set_xticks(plot_params["FLUX_XTICKS"])
    ax.set_yticks(plot_params["IC_YTICKS"])

    beautify_axes(
        ax,
        r"$\Phi/\Phi_0$",
        r"$I_c(\Phi)/I_c(0)$",
        plot_params,
    )

    if plot_params["SHOW_LEGEND"]:
        leg = ax.legend(
            loc=plot_params["LEGEND_LOC"],
            frameon=True,
            fancybox=True,
            framealpha=0.92,
            fontsize=plot_params["LEGEND_SIZE"],
            edgecolor="0.75",
        )
        leg.get_frame().set_linewidth(0.8)

    savefig_no_margin(
        fig,
        os.path.join(plot_params["PLOT_DIR"], "fraunhofer_Ic_vs_flux.png"),
        dpi=plot_params["SAVE_DPI"],
    )

    plt.close(fig)


# ============================================================
# Plot: selected CPR curves
# ============================================================

def plot_selected_cpr(data, params, plot_params):
    ensure_dir(plot_params["PLOT_DIR"])

    flux_values = data["flux_values"]
    phi_values = data["phi_values"]
    I_map = data["I_map"]

    selected_fluxes = plot_params["SELECTED_FLUXES"]
    colors = plot_params["SELECTED_CPR_COLORS"]

    fig, ax = plt.subplots(figsize=plot_params["FIGSIZE_CPR"])

    if plot_params["CPR_NORMALIZATION"] == "global":
        I_scale = np.max(np.abs(I_map)) + 1.0e-30
    elif plot_params["CPR_NORMALIZATION"] == "each_curve":
        I_scale = None
    else:
        raise ValueError("CPR_NORMALIZATION must be 'global' or 'each_curve'.")

    for i, flux_target in enumerate(selected_fluxes):
        color = colors[i % len(colors)]

        idx = nearest_index(flux_values, flux_target)
        flux_actual = flux_values[idx]

        y = I_map[idx].copy()

        if plot_params["CPR_NORMALIZATION"] == "global":
            y = y / I_scale
        else:
            y = y / (np.max(np.abs(y)) + 1.0e-30)

        ax.plot(
            phi_values / np.pi,
            y,
            color=color,
            lw=plot_params["LW_MAIN"],
            label=rf"$\Phi/\Phi_0={flux_actual:.2f}$",
        )

    ax.axhline(0.0, color=plot_params["COLOR_GUIDE"], lw=0.9, ls="--", zorder=0)

    ax.set_xlim(plot_params["PHI_XLIM"])
    set_ticks_from_params(ax, plot_params["PHI_XTICKS"], None)

    if plot_params.get("CPR_YLIM", None) is not None:
        ax.set_ylim(plot_params["CPR_YLIM"])
    if plot_params.get("CPR_YTICKS", None) is not None:
        set_ticks_from_params(ax, None, plot_params["CPR_YTICKS"])

    beautify_axes(
        ax,
        r"$\varphi/\pi$",
        r"$I(\varphi,\Phi)/I_{\max}$",
        plot_params,
    )

    if plot_params["SHOW_LEGEND"]:
        leg = ax.legend(
            loc=plot_params["LEGEND_LOC"],
            frameon=True,
            fancybox=True,
            framealpha=0.92,
            fontsize=plot_params["LEGEND_SIZE"],
            edgecolor="0.75",
        )
        leg.get_frame().set_linewidth(0.8)

    savefig_no_margin(
        fig,
        os.path.join(plot_params["PLOT_DIR"], "selected_CPR_vs_flux.png"),
        dpi=plot_params["SAVE_DPI"],
    )

    plt.close(fig)


# ============================================================
# Plot: current phase-flux map
# ============================================================

def plot_current_map(data, params, plot_params):
    ensure_dir(plot_params["PLOT_DIR"])

    flux_values = data["flux_values"]
    phi_values = data["phi_values"]
    I_map = data["I_map"]

    I_scale = np.max(np.abs(I_map)) + 1.0e-30

    fig, ax = plt.subplots(figsize=plot_params["FIGSIZE_MAP"])

    dphi = phi_values[1] - phi_values[0]

    im = ax.imshow(
        I_map / I_scale,
        origin="lower",
        aspect="auto",
        extent=[
            phi_values[0] / np.pi,
            (phi_values[-1] + dphi) / np.pi,
            flux_values[0],
            flux_values[-1],
        ],
        cmap=plot_params["CURRENT_MAP_CMAP"],
        vmin=plot_params["CURRENT_MAP_VMIN"],
        vmax=plot_params["CURRENT_MAP_VMAX"],
        interpolation=plot_params["MAP_INTERPOLATION"],
    )

    cbar = fig.colorbar(
        im,
        ax=ax,
        pad=plot_params["CBAR_PAD"],
        shrink=plot_params["CBAR_SHRINK"],
        aspect=plot_params["CBAR_ASPECT"],
        fraction=plot_params["CBAR_FRACTION"],
    )
    cbar.set_label(plot_params["CBAR_LABEL"], fontsize=plot_params["CBAR_LABEL_SIZE"])
    cbar.set_ticks(at_most_three_ticks(plot_params["CBAR_TICKS"]))
    cbar.ax.tick_params(
        labelsize=plot_params["CBAR_TICK_SIZE"],
        width=1.0,
        length=4,
    )

    ax.set_xlim(plot_params["MAP_PHI_XLIM"])
    ax.set_ylim(plot_params["MAP_FLUX_YLIM"])
    set_ticks_from_params(ax, plot_params["MAP_PHI_XTICKS"], plot_params["MAP_FLUX_YTICKS"])

    beautify_axes(
        ax,
        r"$\varphi/\pi$",
        r"$\Phi/\Phi_0$",
        plot_params,
    )

    savefig_no_margin(
        fig,
        os.path.join(plot_params["PLOT_DIR"], "current_phase_flux_map.png"),
        dpi=plot_params["SAVE_DPI"],
    )

    plt.close(fig)


# ============================================================
# Plot all figures from saved data
# ============================================================

def plot_all_from_file(data_file, plot_params):
    """
    Plot-only function.

    It reads DATA_FILE and generates all figures. It does not recalculate data.
    """
    data, saved_params = load_saved_data(data_file)

    set_plot_style(plot_params)

    ensure_dir(plot_params["PLOT_DIR"])

    print("=" * 80)
    print(f"Loaded data from: {data_file}")
    print(f"Plot directory:   {plot_params['PLOT_DIR']}")
    print("=" * 80)

    if plot_params["PLOT_GEOMETRY"]:
        print("Plotting geometry preview ...")
        plot_geometry_preview(saved_params, plot_params)

    if plot_params["PLOT_FRAUNHOFER"]:
        print("Plotting Fraunhofer pattern ...")
        plot_fraunhofer(data, saved_params, plot_params)

    if plot_params["PLOT_SELECTED_CPR"]:
        print("Plotting selected CPR curves ...")
        plot_selected_cpr(data, saved_params, plot_params)

    if plot_params["PLOT_CURRENT_MAP"]:
        print("Plotting current phase-flux map ...")
        plot_current_map(data, saved_params, plot_params)

    print("=" * 80)
    print("Plotting done.")
    print("=" * 80)


# ============================================================
# Main control
# ============================================================

if __name__ == "__main__":

    # ========================================================
    # Run switches
    # ========================================================

    RUN_CALCULATION = True
    RUN_PLOTTING = True

    # Set RUN_CALCULATION = False and RUN_PLOTTING = True
    # to redraw figures from existing data without recalculation.

    # ========================================================
    # Calculation parameters
    # ========================================================

    OUTDIR = "sns_2d_swave_fraunhofer"

    DATA_FILE = os.path.join(
        OUTDIR,
        "sns_2d_fraunhofer_data.npz",
    )

    params = dict(
        # ---------------- Output
        OUTDIR=OUTDIR,
        DATA_FILE=DATA_FILE,

        # ---------------- Geometry
        LX_S_LEFT=5,
        LX_N=6,
        LX_S_RIGHT=5,
        WY=24,

        # ---------------- Tight-binding model
        T_HOP=1.0,
        MU_S=-1.0,
        MU_N=-1.0,
        DELTA0=0.25,

        # ---------------- Optional potentials
        U_N=0.0,
        U_INTERFACE=0.0,

        # ---------------- Spin degeneracy
        SPIN_DEGENERACY=2.0,

        # ---------------- Magnetic flux scan
        FLUX_MIN=-5.0,
        FLUX_MAX=5.0,
        N_FLUX=81,

        # ---------------- Josephson phase scan
        PHI_MIN=0.0,
        PHI_MAX=2.0 * np.pi,
        N_PHI=81,

        # ---------------- Numerics
        EIG_CUTOFF=1.0e-10,
        CURRENT_SIGN=1.0,
        SUBTRACT_CURRENT_MEAN=True,
    )

    # ========================================================
    # Plot parameters
    # ========================================================

    plot_params = dict(
        # ---------------- Output
        PLOT_DIR=os.path.join(OUTDIR, "figures"),

        # ---------------- Which plots
        PLOT_GEOMETRY=True,
        PLOT_FRAUNHOFER=True,
        PLOT_SELECTED_CPR=True,
        PLOT_CURRENT_MAP=True,

        # ---------------- General style
        FONT_FAMILY="Times New Roman",
        MATH_FONTSET="stix",
        FIGURE_DPI=180,
        SAVE_DPI=450,

        AXES_LINEWIDTH=1.2,
        TICK_WIDTH=1.2,
        TICK_LENGTH=5,
        LABEL_SIZE=15,
        TICK_SIZE=12,
        LEGEND_SIZE=10.5,
        CBAR_LABEL_SIZE=12,
        CBAR_TICK_SIZE=9,

        LW_MAIN=2.0,
        LW_ANALYTIC=1.4,

        SHOW_LEGEND=True,
        LEGEND_LOC="best",

        # ---------------- Figure sizes
        FIGSIZE_GEOMETRY=(5.8, 2.35),
        FIGSIZE_FRAUNHOFER=(5.3, 3.7),
        FIGSIZE_CPR=(5.3, 3.7),
        FIGSIZE_MAP=(5.6, 3.9),

        # ---------------- Colors and maps
        COLOR_IC="#D62728",
        COLOR_ANALYTIC="0.30",
        COLOR_GUIDE="0.70",
        LABEL_NUMERICAL="BdG",

        GEOMETRY_CMAP="viridis",

        # Density-map style. These can be changed without recalculation.
        CURRENT_MAP_CMAP="bwr",
        MAP_INTERPOLATION="spline16",
        CURRENT_MAP_VMIN=-1.0,
        CURRENT_MAP_VMAX=1.0,
        CBAR_LABEL=r"$I/I_{\max}$",
        CBAR_TICKS=[-1.0, 0.0, 1.0],
        CBAR_PAD=0.015,
        CBAR_SHRINK=0.72,
        CBAR_ASPECT=22,
        CBAR_FRACTION=0.046,

        # ---------------- Geometry plot. At most three ticks are kept.
        GEOMETRY_XTICKS=None,
        GEOMETRY_YTICKS=None,

        # ---------------- Fraunhofer plot. At most three ticks are kept.
        SHOW_ANALYTIC_FRAUNHOFER=True,
        IC_NORMALIZATION="zero_flux",   # "zero_flux" or "max"
        FLUX_XLIM=(-5.0, 5.0),
        IC_YLIM=(-0.02, 1.08),
        FLUX_XTICKS=[-5.0, 0.0, 5.0],
        IC_YTICKS=[0.0, 0.5, 1.0],

        # ---------------- CPR plot. At most three ticks are kept.
        SELECTED_FLUXES=[0.0, 1.0, 2.0],
        SELECTED_CPR_COLORS=[
            "#D62728",
            "#1F77B4",
            "#2CA02C",
        ],
        CPR_NORMALIZATION="global",     # "global" or "each_curve"
        PHI_XLIM=(0.0, 2.0),
        PHI_XTICKS=[0.0, 1.0, 2.0],
        CPR_YLIM=(-1.05, 1.05),
        CPR_YTICKS=[-1.0, 0.0, 1.0],

        # ---------------- Current map. At most three ticks are kept.
        MAP_PHI_XLIM=(0.0, 2.0),
        MAP_PHI_XTICKS=[0.0, 1.0, 2.0],
        MAP_FLUX_YLIM=(-5.0, 5.0),
        MAP_FLUX_YTICKS=[-5.0, 0.0, 5.0],
    )

    # ========================================================
    # Print summary
    # ========================================================

    print("=" * 80)
    print("2D finite-width s-wave SNS junction with orbital magnetic flux")
    print("=" * 80)
    print(f"RUN_CALCULATION = {RUN_CALCULATION}")
    print(f"RUN_PLOTTING    = {RUN_PLOTTING}")
    print(f"OUTDIR          = {OUTDIR}")
    print(f"DATA_FILE       = {DATA_FILE}")
    print("-" * 80)
    print(f"LX_S_LEFT       = {params['LX_S_LEFT']}")
    print(f"LX_N            = {params['LX_N']}")
    print(f"LX_S_RIGHT      = {params['LX_S_RIGHT']}")
    print(f"WY              = {params['WY']}")
    print(f"NSITES          = {nsites(params)}")
    print(f"BdG dimension   = {2 * nsites(params)}")
    print("-" * 80)
    print(f"T_HOP           = {params['T_HOP']}")
    print(f"MU_S            = {params['MU_S']}")
    print(f"MU_N            = {params['MU_N']}")
    print(f"DELTA0          = {params['DELTA0']}")
    print("-" * 80)
    print(
        f"Flux range      = "
        f"[{params['FLUX_MIN']}, {params['FLUX_MAX']}], "
        f"N_FLUX = {params['N_FLUX']}"
    )
    print(f"N_PHI           = {params['N_PHI']}")
    print("=" * 80)

    # ========================================================
    # Run calculation and/or plotting
    # ========================================================

    if RUN_CALCULATION:
        calculate_and_save_data(params)

    if RUN_PLOTTING:
        plot_all_from_file(DATA_FILE, plot_params)

    print("=" * 80)
    print("All tasks finished.")
    print("=" * 80)

2D finite-width s-wave SNS junction with orbital magnetic flux
RUN_CALCULATION = True
RUN_PLOTTING    = True
OUTDIR          = sns_2d_swave_fraunhofer
DATA_FILE       = sns_2d_swave_fraunhofer/sns_2d_fraunhofer_data.npz
--------------------------------------------------------------------------------
LX_S_LEFT       = 5
LX_N            = 6
LX_S_RIGHT      = 5
WY              = 24
NSITES          = 384
BdG dimension   = 768
--------------------------------------------------------------------------------
T_HOP           = 1.0
MU_S            = -1.0
MU_N            = -1.0
DELTA0          = 0.25
--------------------------------------------------------------------------------
Flux range      = [-5.0, 5.0], N_FLUX = 81
N_PHI           = 81
Flux    1/81: Phi/Phi0 = -5.000000
Flux    2/81: Phi/Phi0 = -4.875000
Flux    3/81: Phi/Phi0 = -4.750000
Flux    4/81: Phi/Phi0 = -4.625000
Flux    5/81: Phi/Phi0 = -4.500000
Flux    6/81: Phi/Phi0 = -4.375000
Flux    7/81: Phi/Phi0 = -4.250000
Flux    8/81

## SNS，中间区域为量子自旋Hall绝缘体

In [4]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt

try:
    import scipy.linalg as sla
    HAS_SCIPY_LINALG = True
except Exception:
    HAS_SCIPY_LINALG = False


# ============================================================
# Basic utilities
# ============================================================

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def savefig_no_margin(fig, filename, dpi=450):
    fig.savefig(
        filename,
        dpi=dpi,
        bbox_inches="tight",
        pad_inches=0.0,
    )


def set_plot_style(plot_params):
    plt.rcParams["font.family"] = plot_params["FONT_FAMILY"]
    plt.rcParams["mathtext.fontset"] = plot_params["MATH_FONTSET"]
    plt.rcParams["axes.linewidth"] = plot_params["AXES_LINEWIDTH"]
    plt.rcParams["xtick.direction"] = "in"
    plt.rcParams["ytick.direction"] = "in"
    plt.rcParams["xtick.top"] = True
    plt.rcParams["ytick.right"] = True
    plt.rcParams["figure.dpi"] = plot_params["FIGURE_DPI"]


def beautify_axes(ax, xlabel=None, ylabel=None, plot_params=None):
    if plot_params is None:
        label_size = 15
        tick_size = 12
        axes_lw = 1.2
        tick_lw = 1.2
        tick_len = 5
    else:
        label_size = plot_params["LABEL_SIZE"]
        tick_size = plot_params["TICK_SIZE"]
        axes_lw = plot_params["AXES_LINEWIDTH"]
        tick_lw = plot_params["TICK_WIDTH"]
        tick_len = plot_params["TICK_LENGTH"]

    if xlabel is not None:
        ax.set_xlabel(xlabel, fontsize=label_size)
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontsize=label_size)

    ax.tick_params(labelsize=tick_size, width=tick_lw, length=tick_len)

    for spine in ax.spines.values():
        spine.set_linewidth(axes_lw)


def hermitian_eigvalsh(H):
    if HAS_SCIPY_LINALG:
        return sla.eigvalsh(
            H,
            overwrite_a=True,
            check_finite=False,
            driver="evr",
        )

    return np.linalg.eigvalsh(H)


# ============================================================
# Internal matrices
# Electron basis at each site:
#     (orbital 0 up, orbital 0 down, orbital 1 up, orbital 1 down)
# ============================================================

def pauli_matrices():
    s0 = np.array([[1, 0], [0, 1]], dtype=complex)
    sx = np.array([[0, 1], [1, 0]], dtype=complex)
    sy = np.array([[0, -1j], [1j, 0]], dtype=complex)
    sz = np.array([[1, 0], [0, -1]], dtype=complex)
    return s0, sx, sy, sz


def internal_matrices():
    rho0, rhox, rhoy, rhoz = pauli_matrices()
    spin0, spinx, spiny, spinz = pauli_matrices()

    mats = dict(
        I4=np.eye(4, dtype=complex),

        rho0_s0=np.kron(rho0, spin0),
        rhox_s0=np.kron(rhox, spin0),
        rhoy_s0=np.kron(rhoy, spin0),
        rhoz_s0=np.kron(rhoz, spin0),

        rhox_sz=np.kron(rhox, spinz),
        rho0_sy=np.kron(rho0, spiny),
    )

    # spin-singlet onsite pairing within each orbital:
    # Delta * i s_y, orbital diagonal
    mats["PAIR_SINGLET"] = np.kron(rho0, 1j * spiny)

    return mats


# ============================================================
# Geometry helpers
# ============================================================

def lx_total(params):
    return (
        params["LX_S_LEFT"]
        + params["LX_QSH"]
        + params["LX_S_RIGHT"]
    )


def nsites(params):
    return lx_total(params) * params["WY"]


def electron_dim(params):
    return 4 * nsites(params)


def bdg_dim(params):
    return 2 * electron_dim(params)


def site_index(x, y, params):
    return x * params["WY"] + y


def block_slice(x, y, params):
    i = site_index(x, y, params)
    return slice(4 * i, 4 * i + 4)


def region_of_x(x, params):
    if x < params["LX_S_LEFT"]:
        return "L"
    if x < params["LX_S_LEFT"] + params["LX_QSH"]:
        return "QSH"
    return "R"


def is_interface_bond_x(x, params):
    """
    x-bond connects x -> x+1.

    Interfaces:
        left S / QSH: x = LX_S_LEFT - 1
        QSH / right S: x = LX_S_LEFT + LX_QSH - 1
    """
    return (
        x == params["LX_S_LEFT"] - 1
        or x == params["LX_S_LEFT"] + params["LX_QSH"] - 1
    )


# ============================================================
# Model blocks
# ============================================================

def s_region_onsite(params, mats):
    """
    Ordinary metallic s-wave superconducting region normal-state electron onsite.

    Electron part:
        H_S = -mu_S I

    The pairing is added separately in the BdG off-diagonal block.
    """
    return -params["MU_S"] * mats["I4"]


def s_region_hop_x(params, mats):
    return -params["T_S"] * mats["I4"]


def s_region_hop_y(params, mats):
    return -params["T_S"] * mats["I4"]


def qsh_onsite(params, mats):
    """
    BHZ/QSH onsite block.

    H_QSH(k) =
        [M - 2B(2 - cos kx - cos ky)] rho_z
        + A sin kx rho_x s_z
        + A sin ky rho_y
        - mu_QSH I

    Real-space onsite:
        (M - 4B) rho_z - mu_QSH I
    """
    M = params["M_QSH"]
    B = params["B_QSH"]
    mu = params["MU_QSH"]

    return (M - 4.0 * B) * mats["rhoz_s0"] - mu * mats["I4"]


def qsh_hop_x(params, mats):
    """
    x hopping:
        B rho_z - i A/2 rho_x s_z
    """
    A = params["A_QSH"]
    B = params["B_QSH"]

    return B * mats["rhoz_s0"] - 0.5j * A * mats["rhox_sz"]


def qsh_hop_y(params, mats):
    """
    y hopping:
        B rho_z - i A/2 rho_y
    """
    A = params["A_QSH"]
    B = params["B_QSH"]

    return B * mats["rhoz_s0"] - 0.5j * A * mats["rhoy_s0"]


def interface_hop(params, mats):
    """
    Simple orbital/spin conserving hopping across the S-QSH interface.
    """
    return -params["T_INTERFACE"] * mats["I4"]


def onsite_block(x, params, mats):
    reg = region_of_x(x, params)

    if reg == "QSH":
        return qsh_onsite(params, mats)

    return s_region_onsite(params, mats)


def hop_x_block(x, params, mats):
    """
    Hopping from x to x+1 at fixed y.
    """
    if is_interface_bond_x(x, params):
        return interface_hop(params, mats)

    reg_left = region_of_x(x, params)
    reg_right = region_of_x(x + 1, params)

    if reg_left == "QSH" and reg_right == "QSH":
        return qsh_hop_x(params, mats)

    return s_region_hop_x(params, mats)


def hop_y_block(x, params, mats):
    reg = region_of_x(x, params)

    if reg == "QSH":
        return qsh_hop_y(params, mats)

    return s_region_hop_y(params, mats)


# ============================================================
# Hamiltonian construction
# ============================================================

def build_electron_hamiltonian(params):
    mats = internal_matrices()

    lx = lx_total(params)
    wy = params["WY"]
    edim = electron_dim(params)

    H = np.zeros((edim, edim), dtype=complex)

    # Onsite
    for x in range(lx):
        H0 = onsite_block(x, params, mats)

        for y in range(wy):
            sl = block_slice(x, y, params)
            H[sl, sl] = H0

    # Hopping
    for x in range(lx):
        for y in range(wy):

            # x hopping
            if x + 1 < lx:
                Vx = hop_x_block(x, params, mats)

                sl_i = block_slice(x, y, params)
                sl_j = block_slice(x + 1, y, params)

                H[sl_i, sl_j] = Vx
                H[sl_j, sl_i] = Vx.conjugate().T

            # y hopping
            if y + 1 < wy:
                Vy = hop_y_block(x, params, mats)

                sl_i = block_slice(x, y, params)
                sl_j = block_slice(x, y + 1, params)

                H[sl_i, sl_j] = Vy
                H[sl_j, sl_i] = Vy.conjugate().T

    return H


def build_pairing_matrix(phi, params):
    """
    Onsite spin-singlet s-wave pairing in the left and right S regions.

    Left S:
        Delta_L = Delta0 exp(-i phi/2)

    Right S:
        Delta_R = Delta0 exp(+i phi/2)

    QSH region:
        Delta = 0
    """
    mats = internal_matrices()

    lx = lx_total(params)
    wy = params["WY"]
    edim = electron_dim(params)

    D = np.zeros((edim, edim), dtype=complex)

    pair_unit = mats["PAIR_SINGLET"]

    phase_L = np.exp(-0.5j * phi)
    phase_R = np.exp(+0.5j * phi)

    for x in range(lx):
        reg = region_of_x(x, params)

        if reg == "L":
            delta = params["DELTA0"] * phase_L
        elif reg == "R":
            delta = params["DELTA0"] * phase_R
        else:
            delta = 0.0

        if abs(delta) > 0.0:
            block = delta * pair_unit

            for y in range(wy):
                sl = block_slice(x, y, params)
                D[sl, sl] = block

    return D


def build_bdg_hamiltonian(He, phi, params):
    """
    BdG basis:
        (all electrons, all holes)

    H_BdG =
        [ H_e          Delta      ]
        [ Delta^dag   -H_e^*     ]
    """
    edim = He.shape[0]
    dim = 2 * edim

    D = build_pairing_matrix(phi, params)

    Hbdg = np.zeros((dim, dim), dtype=complex)

    Hbdg[:edim, :edim] = He
    Hbdg[:edim, edim:] = D
    Hbdg[edim:, :edim] = D.conjugate().T
    Hbdg[edim:, edim:] = -He.conjugate()

    return Hbdg


# ============================================================
# Josephson energy, CPR, and spectrum
# ============================================================

def diagonalize_bdg_for_phi(He, phi, params):
    Hbdg = build_bdg_hamiltonian(He, phi, params)
    evals = hermitian_eigvalsh(Hbdg)
    return evals


def ground_state_energy_from_evals(evals, params):
    """
    Zero-temperature BdG ground-state energy.

    Up to phi-independent constants:
        F(phi) = -1/2 sum_{E_n > 0} E_n
    """
    positive = evals[evals > params["EIG_CUTOFF"]]
    return float(np.real(-0.5 * np.sum(positive)))


def periodic_derivative(values, dx):
    return (np.roll(values, -1) - np.roll(values, 1)) / (2.0 * dx)


def calculate_cpr(params):
    """
    Calculate:
        F(phi)
        I(phi) = (2e/hbar) dF/dphi
        lowest positive BdG energies vs phi

    Here e = hbar = 1, so I = 2 dF/dphi.
    """
    phi_values = np.linspace(
        params["PHI_MIN"],
        params["PHI_MAX"],
        params["N_PHI"],
        endpoint=False,
    )

    He = build_electron_hamiltonian(params)

    F_values = np.zeros_like(phi_values, dtype=float)

    n_spec = params["N_SPEC_EIGS"]
    low_positive = np.zeros((len(phi_values), n_spec), dtype=float)

    for ip, phi in enumerate(phi_values):
        print(f"phi {ip + 1:4d}/{len(phi_values)}: phi/pi = {phi / np.pi:.6f}")

        evals = diagonalize_bdg_for_phi(He, phi, params)

        F_values[ip] = ground_state_energy_from_evals(evals, params)

        pos = evals[evals > params["EIG_CUTOFF"]]
        if len(pos) >= n_spec:
            low_positive[ip, :] = pos[:n_spec]
        else:
            low_positive[ip, :len(pos)] = pos
            low_positive[ip, len(pos):] = np.nan

    dphi = phi_values[1] - phi_values[0]

    I_values = (
        params["CURRENT_SIGN"]
        * 2.0
        * periodic_derivative(F_values, dphi)
    )

    if params["SUBTRACT_CURRENT_MEAN"]:
        I_values = I_values - np.mean(I_values)

    Ic_plus = np.max(I_values)
    Ic_minus = np.min(I_values)
    Ic_abs = np.max(np.abs(I_values))

    return dict(
        phi_values=phi_values,
        F_values=F_values,
        I_values=I_values,
        low_positive=low_positive,
        Ic_plus=Ic_plus,
        Ic_minus=Ic_minus,
        Ic_abs=Ic_abs,
    )


# ============================================================
# Calculation and saving
# ============================================================

def calculate_and_save_data(params):
    ensure_dir(params["OUTDIR"])

    result = calculate_cpr(params)

    params_json = json.dumps(params, indent=2, sort_keys=True)

    np.savez_compressed(
        params["DATA_FILE"],
        phi_values=result["phi_values"],
        F_values=result["F_values"],
        I_values=result["I_values"],
        low_positive=result["low_positive"],
        Ic_plus=np.array(result["Ic_plus"]),
        Ic_minus=np.array(result["Ic_minus"]),
        Ic_abs=np.array(result["Ic_abs"]),
        params_json=np.array(params_json),
    )

    print("=" * 80)
    print(f"Saved data to: {params['DATA_FILE']}")
    print("=" * 80)

    return params["DATA_FILE"]


# ============================================================
# Loading saved data
# ============================================================

def load_saved_data(data_file):
    loaded = np.load(data_file, allow_pickle=True)

    data = {key: loaded[key] for key in loaded.files}

    if "params_json" in data:
        params_json = data["params_json"].item()
        params = json.loads(params_json)
    else:
        params = {}

    return data, params


# ============================================================
# Plotting
# ============================================================

def plot_geometry_preview(params, plot_params):
    ensure_dir(plot_params["PLOT_DIR"])

    lx = lx_total(params)
    wy = params["WY"]

    region_map = np.zeros((wy, lx))

    for x in range(lx):
        reg = region_of_x(x, params)

        if reg == "L":
            val = 0.0
        elif reg == "QSH":
            val = 0.5
        else:
            val = 1.0

        region_map[:, x] = val

    fig, ax = plt.subplots(figsize=plot_params["FIGSIZE_GEOMETRY"])

    ax.imshow(
        region_map,
        origin="lower",
        aspect="auto",
        cmap=plot_params["GEOMETRY_CMAP"],
        interpolation="nearest",
        extent=[0, lx, 0, wy],
    )

    ax.axvline(params["LX_S_LEFT"], color="w", lw=1.4, ls="--")
    ax.axvline(params["LX_S_LEFT"] + params["LX_QSH"], color="w", lw=1.4, ls="--")

    ax.text(
        0.5 * params["LX_S_LEFT"],
        0.5 * wy,
        "S",
        color="w",
        ha="center",
        va="center",
        fontsize=16,
    )

    ax.text(
        params["LX_S_LEFT"] + 0.5 * params["LX_QSH"],
        0.5 * wy,
        "QSH",
        color="w",
        ha="center",
        va="center",
        fontsize=16,
    )

    ax.text(
        params["LX_S_LEFT"] + params["LX_QSH"] + 0.5 * params["LX_S_RIGHT"],
        0.5 * wy,
        "S",
        color="w",
        ha="center",
        va="center",
        fontsize=16,
    )

    beautify_axes(
        ax,
        r"$x$",
        r"$y$",
        plot_params,
    )

    savefig_no_margin(
        fig,
        os.path.join(plot_params["PLOT_DIR"], "geometry_preview_QSH_SNS.png"),
        dpi=plot_params["SAVE_DPI"],
    )

    plt.close(fig)


def plot_cpr(data, params, plot_params):
    ensure_dir(plot_params["PLOT_DIR"])

    phi = data["phi_values"]
    I = data["I_values"]

    if plot_params["NORMALIZE_CURRENT"]:
        y = I / (np.max(np.abs(I)) + 1.0e-30)
        ylabel = r"$I(\varphi)/I_{\max}$"
    else:
        y = I
        ylabel = r"$I(\varphi)$"

    fig, ax = plt.subplots(figsize=plot_params["FIGSIZE_CPR"])

    ax.plot(
        phi / np.pi,
        y,
        color=plot_params["COLOR_CURRENT"],
        lw=plot_params["LW_MAIN"],
    )

    ax.axhline(0.0, color=plot_params["COLOR_GUIDE"], lw=0.9, ls="--", zorder=0)

    ax.set_xlim(plot_params["PHI_XLIM"])
    ax.set_xticks(plot_params["PHI_XTICKS"])

    if plot_params["NORMALIZE_CURRENT"]:
        ax.set_ylim(plot_params["CURRENT_NORM_YLIM"])
        ax.set_yticks(plot_params["CURRENT_NORM_YTICKS"])

    beautify_axes(
        ax,
        r"$\varphi/\pi$",
        ylabel,
        plot_params,
    )

    savefig_no_margin(
        fig,
        os.path.join(plot_params["PLOT_DIR"], "QSH_SNS_CPR.png"),
        dpi=plot_params["SAVE_DPI"],
    )

    plt.close(fig)


def plot_free_energy(data, params, plot_params):
    ensure_dir(plot_params["PLOT_DIR"])

    phi = data["phi_values"]
    F = data["F_values"]

    # subtract mean for display
    F_disp = F - np.mean(F)

    fig, ax = plt.subplots(figsize=plot_params["FIGSIZE_ENERGY"])

    ax.plot(
        phi / np.pi,
        F_disp,
        color=plot_params["COLOR_FREE_ENERGY"],
        lw=plot_params["LW_MAIN"],
    )

    ax.axhline(0.0, color=plot_params["COLOR_GUIDE"], lw=0.9, ls="--", zorder=0)

    ax.set_xlim(plot_params["PHI_XLIM"])
    ax.set_xticks(plot_params["PHI_XTICKS"])

    beautify_axes(
        ax,
        r"$\varphi/\pi$",
        r"$F(\varphi)-\langle F\rangle$",
        plot_params,
    )

    savefig_no_margin(
        fig,
        os.path.join(plot_params["PLOT_DIR"], "QSH_SNS_free_energy.png"),
        dpi=plot_params["SAVE_DPI"],
    )

    plt.close(fig)


def plot_low_energy_spectrum(data, params, plot_params):
    ensure_dir(plot_params["PLOT_DIR"])

    phi = data["phi_values"]
    low_positive = data["low_positive"]

    fig, ax = plt.subplots(figsize=plot_params["FIGSIZE_SPECTRUM"])

    for n in range(low_positive.shape[1]):
        ax.plot(
            phi / np.pi,
            low_positive[:, n],
            color=plot_params["COLOR_SPECTRUM"],
            lw=plot_params["LW_SPECTRUM"],
            alpha=plot_params["ALPHA_SPECTRUM"],
        )

    ax.axhline(0.0, color=plot_params["COLOR_GUIDE"], lw=0.9, ls="--", zorder=0)

    ax.set_xlim(plot_params["PHI_XLIM"])
    ax.set_xticks(plot_params["PHI_XTICKS"])

    if plot_params["SPECTRUM_YLIM"] is not None:
        ax.set_ylim(plot_params["SPECTRUM_YLIM"])

    beautify_axes(
        ax,
        r"$\varphi/\pi$",
        r"$E_n(\varphi)$",
        plot_params,
    )

    savefig_no_margin(
        fig,
        os.path.join(plot_params["PLOT_DIR"], "QSH_SNS_low_energy_spectrum.png"),
        dpi=plot_params["SAVE_DPI"],
    )

    plt.close(fig)


def plot_combined(data, params, plot_params):
    ensure_dir(plot_params["PLOT_DIR"])

    phi = data["phi_values"]
    I = data["I_values"]
    low_positive = data["low_positive"]

    if plot_params["NORMALIZE_CURRENT"]:
        I_plot = I / (np.max(np.abs(I)) + 1.0e-30)
        ylabel_I = r"$I/I_{\max}$"
    else:
        I_plot = I
        ylabel_I = r"$I$"

    fig, axes = plt.subplots(
        1,
        2,
        figsize=plot_params["FIGSIZE_COMBINED"],
        gridspec_kw=dict(wspace=0.32),
    )

    ax = axes[0]

    ax.plot(
        phi / np.pi,
        I_plot,
        color=plot_params["COLOR_CURRENT"],
        lw=plot_params["LW_MAIN"],
    )

    ax.axhline(0.0, color=plot_params["COLOR_GUIDE"], lw=0.9, ls="--", zorder=0)

    ax.set_xlim(plot_params["PHI_XLIM"])
    ax.set_xticks(plot_params["PHI_XTICKS"])

    if plot_params["NORMALIZE_CURRENT"]:
        ax.set_ylim(plot_params["CURRENT_NORM_YLIM"])
        ax.set_yticks(plot_params["CURRENT_NORM_YTICKS"])

    beautify_axes(
        ax,
        r"$\varphi/\pi$",
        ylabel_I,
        plot_params,
    )

    ax = axes[1]

    for n in range(low_positive.shape[1]):
        ax.plot(
            phi / np.pi,
            low_positive[:, n],
            color=plot_params["COLOR_SPECTRUM"],
            lw=plot_params["LW_SPECTRUM"],
            alpha=plot_params["ALPHA_SPECTRUM"],
        )

    ax.axhline(0.0, color=plot_params["COLOR_GUIDE"], lw=0.9, ls="--", zorder=0)

    ax.set_xlim(plot_params["PHI_XLIM"])
    ax.set_xticks(plot_params["PHI_XTICKS"])

    if plot_params["SPECTRUM_YLIM"] is not None:
        ax.set_ylim(plot_params["SPECTRUM_YLIM"])

    beautify_axes(
        ax,
        r"$\varphi/\pi$",
        r"$E_n(\varphi)$",
        plot_params,
    )

    savefig_no_margin(
        fig,
        os.path.join(plot_params["PLOT_DIR"], "QSH_SNS_CPR_and_spectrum.png"),
        dpi=plot_params["SAVE_DPI"],
    )

    plt.close(fig)


def plot_all_from_file(data_file, plot_params):
    data, saved_params = load_saved_data(data_file)

    set_plot_style(plot_params)
    ensure_dir(plot_params["PLOT_DIR"])

    print("=" * 80)
    print(f"Loaded data from: {data_file}")
    print(f"Plot directory:   {plot_params['PLOT_DIR']}")
    print("=" * 80)

    if plot_params["PLOT_GEOMETRY"]:
        print("Plotting geometry preview ...")
        plot_geometry_preview(saved_params, plot_params)

    if plot_params["PLOT_CPR"]:
        print("Plotting CPR ...")
        plot_cpr(data, saved_params, plot_params)

    if plot_params["PLOT_FREE_ENERGY"]:
        print("Plotting free energy ...")
        plot_free_energy(data, saved_params, plot_params)

    if plot_params["PLOT_LOW_ENERGY_SPECTRUM"]:
        print("Plotting low-energy spectrum ...")
        plot_low_energy_spectrum(data, saved_params, plot_params)

    if plot_params["PLOT_COMBINED"]:
        print("Plotting combined figure ...")
        plot_combined(data, saved_params, plot_params)

    print("=" * 80)
    print("Plotting done.")
    print("=" * 80)


# ============================================================
# Main control
# ============================================================

if __name__ == "__main__":

    # ========================================================
    # Run switches
    # ========================================================

    RUN_CALCULATION = True
    RUN_PLOTTING = True

    # ========================================================
    # Calculation parameters
    # ========================================================

    OUTDIR = "sns_QSH_swave_josephson"

    DATA_FILE = os.path.join(
        OUTDIR,
        "QSH_SNS_josephson_data.npz",
    )

    params = dict(
        # ---------------- Output
        OUTDIR=OUTDIR,
        DATA_FILE=DATA_FILE,

        # ---------------- Geometry
        LX_S_LEFT=5,
        LX_QSH=18,
        LX_S_RIGHT=5,
        WY=20,

        # ---------------- s-wave superconducting leads
        T_S=1.0,
        MU_S=0.0,
        DELTA0=0.20,

        # ---------------- S-QSH interface
        T_INTERFACE=0.65,

        # ---------------- QSH/BHZ region
        A_QSH=1.0,
        B_QSH=1.0,
        M_QSH=1.0,
        MU_QSH=0.0,

        # ---------------- Josephson phase scan
        PHI_MIN=0.0,
        PHI_MAX=2.0 * np.pi,
        N_PHI=121,

        # ---------------- Numerics
        EIG_CUTOFF=1.0e-10,
        N_SPEC_EIGS=32,
        CURRENT_SIGN=1.0,
        SUBTRACT_CURRENT_MEAN=True,
    )

    # ========================================================
    # Plot parameters
    # ========================================================

    plot_params = dict(
        # ---------------- Output
        PLOT_DIR=os.path.join(OUTDIR, "figures"),

        # ---------------- Which plots
        PLOT_GEOMETRY=True,
        PLOT_CPR=True,
        PLOT_FREE_ENERGY=True,
        PLOT_LOW_ENERGY_SPECTRUM=True,
        PLOT_COMBINED=True,

        # ---------------- General style
        FONT_FAMILY="Times New Roman",
        MATH_FONTSET="stix",
        FIGURE_DPI=180,
        SAVE_DPI=450,

        AXES_LINEWIDTH=1.2,
        TICK_WIDTH=1.2,
        TICK_LENGTH=5,
        LABEL_SIZE=15,
        TICK_SIZE=12,

        LW_MAIN=2.0,
        LW_SPECTRUM=0.85,
        ALPHA_SPECTRUM=0.70,

        # ---------------- Figure sizes
        FIGSIZE_GEOMETRY=(6.4, 2.6),
        FIGSIZE_CPR=(5.4, 3.8),
        FIGSIZE_ENERGY=(5.4, 3.8),
        FIGSIZE_SPECTRUM=(5.4, 3.8),
        FIGSIZE_COMBINED=(9.6, 3.8),

        # ---------------- Colors and maps
        GEOMETRY_CMAP="viridis",
        COLOR_CURRENT="#D62728",
        COLOR_FREE_ENERGY="#1F77B4",
        COLOR_SPECTRUM="#1F1F1F",
        COLOR_GUIDE="0.70",

        # ---------------- Axes controls
        PHI_XLIM=(0.0, 2.0),
        PHI_XTICKS=[0.0, 0.5, 1.0, 1.5, 2.0],

        NORMALIZE_CURRENT=True,
        CURRENT_NORM_YLIM=(-1.10, 1.10),
        CURRENT_NORM_YTICKS=[-1.0, -0.5, 0.0, 0.5, 1.0],

        SPECTRUM_YLIM=(0.0, 0.35),
    )

    # ========================================================
    # Print summary
    # ========================================================

    print("=" * 80)
    print("Finite-width s-wave / QSH / s-wave Josephson junction")
    print("=" * 80)
    print(f"RUN_CALCULATION = {RUN_CALCULATION}")
    print(f"RUN_PLOTTING    = {RUN_PLOTTING}")
    print(f"OUTDIR          = {OUTDIR}")
    print(f"DATA_FILE       = {DATA_FILE}")
    print(f"HAS_SCIPY_LINALG = {HAS_SCIPY_LINALG}")
    print("-" * 80)
    print(f"LX_S_LEFT       = {params['LX_S_LEFT']}")
    print(f"LX_QSH          = {params['LX_QSH']}")
    print(f"LX_S_RIGHT      = {params['LX_S_RIGHT']}")
    print(f"WY              = {params['WY']}")
    print(f"NSITES          = {nsites(params)}")
    print(f"Electron dim    = {electron_dim(params)}")
    print(f"BdG dim         = {bdg_dim(params)}")
    print("-" * 80)
    print(f"QSH parameters  = A={params['A_QSH']}, B={params['B_QSH']}, M={params['M_QSH']}")
    print(f"MU_QSH          = {params['MU_QSH']}")
    print(f"DELTA0          = {params['DELTA0']}")
    print(f"T_INTERFACE     = {params['T_INTERFACE']}")
    print(f"N_PHI           = {params['N_PHI']}")
    print("=" * 80)

    # ========================================================
    # Run
    # ========================================================

    if RUN_CALCULATION:
        calculate_and_save_data(params)

    if RUN_PLOTTING:
        plot_all_from_file(DATA_FILE, plot_params)

    print("=" * 80)
    print("All tasks finished.")
    print("=" * 80)

Finite-width s-wave / QSH / s-wave Josephson junction
RUN_CALCULATION = True
RUN_PLOTTING    = True
OUTDIR          = sns_QSH_swave_josephson
DATA_FILE       = sns_QSH_swave_josephson/QSH_SNS_josephson_data.npz
HAS_SCIPY_LINALG = True
--------------------------------------------------------------------------------
LX_S_LEFT       = 5
LX_QSH          = 18
LX_S_RIGHT      = 5
WY              = 20
NSITES          = 560
Electron dim    = 2240
BdG dim         = 4480
--------------------------------------------------------------------------------
QSH parameters  = A=1.0, B=1.0, M=1.0
MU_QSH          = 0.0
DELTA0          = 0.2
T_INTERFACE     = 0.65
N_PHI           = 121
phi    1/121: phi/pi = 0.000000
phi    2/121: phi/pi = 0.016529
phi    3/121: phi/pi = 0.033058
phi    4/121: phi/pi = 0.049587
phi    5/121: phi/pi = 0.066116
phi    6/121: phi/pi = 0.082645
phi    7/121: phi/pi = 0.099174
phi    8/121: phi/pi = 0.115702
phi    9/121: phi/pi = 0.132231
phi   10/121: phi/pi = 0.148760
phi   

### 加入磁通调制

In [5]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt

try:
    import scipy.linalg as sla
    HAS_SCIPY_LINALG = True
except Exception:
    HAS_SCIPY_LINALG = False


# ============================================================
# Basic utilities
# ============================================================

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def savefig_no_margin(fig, filename, dpi=450):
    fig.savefig(
        filename,
        dpi=dpi,
        bbox_inches="tight",
        pad_inches=0.0,
    )


def set_plot_style(plot_params):
    plt.rcParams["font.family"] = plot_params["FONT_FAMILY"]
    plt.rcParams["mathtext.fontset"] = plot_params["MATH_FONTSET"]
    plt.rcParams["axes.linewidth"] = plot_params["AXES_LINEWIDTH"]
    plt.rcParams["xtick.direction"] = "in"
    plt.rcParams["ytick.direction"] = "in"
    plt.rcParams["xtick.top"] = True
    plt.rcParams["ytick.right"] = True
    plt.rcParams["figure.dpi"] = plot_params["FIGURE_DPI"]


def beautify_axes(ax, xlabel=None, ylabel=None, plot_params=None):
    if plot_params is None:
        label_size = 15
        tick_size = 12
        axes_lw = 1.2
        tick_lw = 1.2
        tick_len = 5
    else:
        label_size = plot_params["LABEL_SIZE"]
        tick_size = plot_params["TICK_SIZE"]
        axes_lw = plot_params["AXES_LINEWIDTH"]
        tick_lw = plot_params["TICK_WIDTH"]
        tick_len = plot_params["TICK_LENGTH"]

    if xlabel is not None:
        ax.set_xlabel(xlabel, fontsize=label_size)
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontsize=label_size)

    ax.tick_params(labelsize=tick_size, width=tick_lw, length=tick_len)

    for spine in ax.spines.values():
        spine.set_linewidth(axes_lw)


def hermitian_eigvalsh(H):
    if HAS_SCIPY_LINALG:
        return sla.eigvalsh(
            H,
            overwrite_a=True,
            check_finite=False,
            driver="evr",
        )

    return np.linalg.eigvalsh(H)


# ============================================================
# Internal matrices
# Electron basis:
#     (orbital 0 up, orbital 0 down, orbital 1 up, orbital 1 down)
# ============================================================

def pauli_matrices():
    s0 = np.array([[1, 0], [0, 1]], dtype=complex)
    sx = np.array([[0, 1], [1, 0]], dtype=complex)
    sy = np.array([[0, -1j], [1j, 0]], dtype=complex)
    sz = np.array([[1, 0], [0, -1]], dtype=complex)
    return s0, sx, sy, sz


def internal_matrices():
    rho0, rhox, rhoy, rhoz = pauli_matrices()
    spin0, spinx, spiny, spinz = pauli_matrices()

    mats = dict(
        I4=np.eye(4, dtype=complex),

        rho0_s0=np.kron(rho0, spin0),
        rhox_s0=np.kron(rhox, spin0),
        rhoy_s0=np.kron(rhoy, spin0),
        rhoz_s0=np.kron(rhoz, spin0),

        rhox_sz=np.kron(rhox, spinz),
    )

    # Orbital-diagonal spin-singlet pairing:
    # Delta * i s_y
    mats["PAIR_SINGLET"] = np.kron(rho0, 1j * spiny)

    return mats


# ============================================================
# Geometry helpers
# ============================================================

def lx_total(params):
    return (
        params["LX_S_LEFT"]
        + params["LX_QSH"]
        + params["LX_S_RIGHT"]
    )


def nsites(params):
    return lx_total(params) * params["WY"]


def electron_dim(params):
    return 4 * nsites(params)


def bdg_dim(params):
    return 2 * electron_dim(params)


def site_index(x, y, params):
    return x * params["WY"] + y


def block_slice(x, y, params):
    i = site_index(x, y, params)
    return slice(4 * i, 4 * i + 4)


def region_of_x(x, params):
    if x < params["LX_S_LEFT"]:
        return "L"
    if x < params["LX_S_LEFT"] + params["LX_QSH"]:
        return "QSH"
    return "R"


def is_interface_bond_x(x, params):
    """
    x-bond connects x -> x+1.

    Interfaces:
        left S / QSH: x = LX_S_LEFT - 1
        QSH / right S: x = LX_S_LEFT + LX_QSH - 1
    """
    return (
        x == params["LX_S_LEFT"] - 1
        or x == params["LX_S_LEFT"] + params["LX_QSH"] - 1
    )


def is_flux_active_x_bond(x, params):
    """
    Peierls phase is applied across the QSH weak-link region,
    including the two S-QSH interface bonds.

    Active x bonds:
        x = LX_S_LEFT - 1, ..., LX_S_LEFT + LX_QSH - 1
    """
    return (
        params["LX_S_LEFT"] - 1
        <= x
        <= params["LX_S_LEFT"] + params["LX_QSH"] - 1
    )


def number_of_flux_bonds(params):
    return params["LX_QSH"] + 1


# ============================================================
# Peierls phase for SQUID-like orbital flux
# ============================================================

def peierls_phase_x(y, flux_phi0, params):
    """
    Electron Peierls phase for one x-direction hopping.

    Gauge:
        A_x = -B (y - y0)

    flux_phi0 is Phi/Phi0, where Phi0 = h/(2e).

    Electron charge uses h/e = 2 Phi0, so the electron Peierls phase
    contains a prefactor pi rather than 2*pi.

    The phase is distributed uniformly over active x bonds. This gives
    opposite phase shifts for the upper and lower helical edge channels.
    """
    wy = params["WY"]

    if wy <= 1:
        return 0.0

    y0 = 0.5 * (wy - 1)
    nxb = number_of_flux_bonds(params)

    flux_per_cell_phi0 = flux_phi0 / (nxb * (wy - 1))

    theta = -np.pi * flux_per_cell_phi0 * (y - y0)

    return theta


# ============================================================
# Model blocks
# ============================================================

def s_region_onsite(params, mats):
    return -params["MU_S"] * mats["I4"]


def s_region_hop_x(params, mats):
    return -params["T_S"] * mats["I4"]


def s_region_hop_y(params, mats):
    return -params["T_S"] * mats["I4"]


def qsh_onsite(params, mats):
    """
    BHZ / QSH model:

    H(k) =
        [M - 2B(2 - cos kx - cos ky)] rho_z
        + A sin kx rho_x s_z
        + A sin ky rho_y
        - mu I

    Real-space onsite:
        (M - 4B) rho_z - mu I
    """
    M = params["M_QSH"]
    B = params["B_QSH"]
    mu = params["MU_QSH"]

    return (M - 4.0 * B) * mats["rhoz_s0"] - mu * mats["I4"]


def qsh_hop_x(params, mats):
    """
    x hopping:
        B rho_z - i A/2 rho_x s_z
    """
    A = params["A_QSH"]
    B = params["B_QSH"]

    return B * mats["rhoz_s0"] - 0.5j * A * mats["rhox_sz"]


def qsh_hop_y(params, mats):
    """
    y hopping:
        B rho_z - i A/2 rho_y
    """
    A = params["A_QSH"]
    B = params["B_QSH"]

    return B * mats["rhoz_s0"] - 0.5j * A * mats["rhoy_s0"]


def interface_hop(params, mats):
    return -params["T_INTERFACE"] * mats["I4"]


def onsite_block(x, params, mats):
    reg = region_of_x(x, params)

    if reg == "QSH":
        return qsh_onsite(params, mats)

    return s_region_onsite(params, mats)


def hop_x_block(x, params, mats):
    if is_interface_bond_x(x, params):
        return interface_hop(params, mats)

    reg_left = region_of_x(x, params)
    reg_right = region_of_x(x + 1, params)

    if reg_left == "QSH" and reg_right == "QSH":
        return qsh_hop_x(params, mats)

    return s_region_hop_x(params, mats)


def hop_y_block(x, params, mats):
    reg = region_of_x(x, params)

    if reg == "QSH":
        return qsh_hop_y(params, mats)

    return s_region_hop_y(params, mats)


# ============================================================
# Hamiltonian construction
# ============================================================

def build_electron_hamiltonian(flux_phi0, params):
    mats = internal_matrices()

    lx = lx_total(params)
    wy = params["WY"]
    edim = electron_dim(params)

    H = np.zeros((edim, edim), dtype=complex)

    # Onsite blocks
    for x in range(lx):
        H0 = onsite_block(x, params, mats)

        for y in range(wy):
            sl = block_slice(x, y, params)
            H[sl, sl] = H0

    # Hopping blocks
    for x in range(lx):
        for y in range(wy):

            # x hopping
            if x + 1 < lx:
                Vx = hop_x_block(x, params, mats)

                if is_flux_active_x_bond(x, params):
                    theta = peierls_phase_x(y, flux_phi0, params)
                    Vx = Vx * np.exp(1j * theta)

                sl_i = block_slice(x, y, params)
                sl_j = block_slice(x + 1, y, params)

                H[sl_i, sl_j] = Vx
                H[sl_j, sl_i] = Vx.conjugate().T

            # y hopping
            if y + 1 < wy:
                Vy = hop_y_block(x, params, mats)

                sl_i = block_slice(x, y, params)
                sl_j = block_slice(x, y + 1, params)

                H[sl_i, sl_j] = Vy
                H[sl_j, sl_i] = Vy.conjugate().T

    return H


def build_pairing_matrix(phi, params):
    mats = internal_matrices()

    lx = lx_total(params)
    wy = params["WY"]
    edim = electron_dim(params)

    D = np.zeros((edim, edim), dtype=complex)

    pair_unit = mats["PAIR_SINGLET"]

    phase_L = np.exp(-0.5j * phi)
    phase_R = np.exp(+0.5j * phi)

    for x in range(lx):
        reg = region_of_x(x, params)

        if reg == "L":
            delta = params["DELTA0"] * phase_L
        elif reg == "R":
            delta = params["DELTA0"] * phase_R
        else:
            delta = 0.0

        if abs(delta) > 0.0:
            block = delta * pair_unit

            for y in range(wy):
                sl = block_slice(x, y, params)
                D[sl, sl] = block

    return D


def build_bdg_hamiltonian(He, phi, params):
    edim = He.shape[0]
    dim = 2 * edim

    D = build_pairing_matrix(phi, params)

    Hbdg = np.zeros((dim, dim), dtype=complex)

    Hbdg[:edim, :edim] = He
    Hbdg[:edim, edim:] = D
    Hbdg[edim:, :edim] = D.conjugate().T
    Hbdg[edim:, edim:] = -He.conjugate()

    return Hbdg


# ============================================================
# Josephson energy and current
# ============================================================

def ground_state_energy_from_evals(evals, params):
    """
    Zero-temperature BdG ground-state energy.

    Up to phi-independent constants:
        F(phi) = -1/2 sum_{E_n > 0} E_n
    """
    positive = evals[evals > params["EIG_CUTOFF"]]
    return float(np.real(-0.5 * np.sum(positive)))


def diagonalize_bdg(He, phi, params):
    Hbdg = build_bdg_hamiltonian(He, phi, params)
    evals = hermitian_eigvalsh(Hbdg)
    return evals


def periodic_derivative(values, dx):
    return (np.roll(values, -1) - np.roll(values, 1)) / (2.0 * dx)


def calculate_cpr_for_flux(flux_phi0, phi_values, params):
    """
    Calculate F(phi), I(phi), and low-energy spectrum for one flux value.
    """
    He = build_electron_hamiltonian(flux_phi0, params)

    n_phi = len(phi_values)
    n_spec = params["N_SPEC_EIGS"]

    F_values = np.zeros(n_phi, dtype=float)
    low_positive = np.zeros((n_phi, n_spec), dtype=float)

    for ip, phi in enumerate(phi_values):
        evals = diagonalize_bdg(He, phi, params)

        F_values[ip] = ground_state_energy_from_evals(evals, params)

        pos = evals[evals > params["EIG_CUTOFF"]]
        if len(pos) >= n_spec:
            low_positive[ip, :] = pos[:n_spec]
        else:
            low_positive[ip, :len(pos)] = pos
            low_positive[ip, len(pos):] = np.nan

    dphi = phi_values[1] - phi_values[0]

    I_values = (
        params["CURRENT_SIGN"]
        * 2.0
        * periodic_derivative(F_values, dphi)
    )

    if params["SUBTRACT_CURRENT_MEAN"]:
        I_values = I_values - np.mean(I_values)

    return F_values, I_values, low_positive


# ============================================================
# Calculation and saving
# ============================================================

def make_grids(params):
    flux_values = np.linspace(
        params["FLUX_MIN"],
        params["FLUX_MAX"],
        params["N_FLUX"],
    )

    phi_values = np.linspace(
        params["PHI_MIN"],
        params["PHI_MAX"],
        params["N_PHI"],
        endpoint=False,
    )

    return flux_values, phi_values


def calculate_and_save_data(params):
    """
    Calculate and save all data. No plotting here.
    """
    ensure_dir(params["OUTDIR"])

    flux_values, phi_values = make_grids(params)

    n_flux = len(flux_values)
    n_phi = len(phi_values)
    n_spec = params["N_SPEC_EIGS"]

    F_map = np.zeros((n_flux, n_phi), dtype=float)
    I_map = np.zeros((n_flux, n_phi), dtype=float)
    low_positive_map = np.zeros((n_flux, n_phi, n_spec), dtype=float)

    Ic_abs = np.zeros(n_flux, dtype=float)
    Ic_plus = np.zeros(n_flux, dtype=float)
    Ic_minus = np.zeros(n_flux, dtype=float)

    for iF, flux in enumerate(flux_values):
        print(
            f"Flux {iF + 1:4d}/{n_flux}: "
            f"Phi/Phi0 = {flux: .6f}"
        )

        F_values, I_values, low_positive = calculate_cpr_for_flux(
            flux_phi0=flux,
            phi_values=phi_values,
            params=params,
        )

        F_map[iF, :] = F_values
        I_map[iF, :] = I_values
        low_positive_map[iF, :, :] = low_positive

        Ic_plus[iF] = np.max(I_values)
        Ic_minus[iF] = np.min(I_values)
        Ic_abs[iF] = np.max(np.abs(I_values))

    params_json = json.dumps(params, indent=2, sort_keys=True)

    np.savez_compressed(
        params["DATA_FILE"],
        flux_values=flux_values,
        phi_values=phi_values,
        F_map=F_map,
        I_map=I_map,
        low_positive_map=low_positive_map,
        Ic_abs=Ic_abs,
        Ic_plus=Ic_plus,
        Ic_minus=Ic_minus,
        params_json=np.array(params_json),
    )

    print("=" * 80)
    print(f"Saved data to: {params['DATA_FILE']}")
    print("=" * 80)

    return params["DATA_FILE"]


# ============================================================
# Loading saved data
# ============================================================

def load_saved_data(data_file):
    loaded = np.load(data_file, allow_pickle=True)

    data = {key: loaded[key] for key in loaded.files}

    if "params_json" in data:
        params_json = data["params_json"].item()
        params = json.loads(params_json)
    else:
        params = {}

    return data, params


# ============================================================
# Plot helpers
# ============================================================

def nearest_index(array, value):
    return int(np.argmin(np.abs(array - value)))


def normalize_ic(flux_values, Ic_abs, plot_params):
    mode = plot_params["IC_NORMALIZATION"]

    if mode == "max":
        return Ic_abs / (np.max(Ic_abs) + 1.0e-30)

    if mode == "zero_flux":
        idx0 = nearest_index(flux_values, 0.0)
        return Ic_abs / (Ic_abs[idx0] + 1.0e-30)

    raise ValueError("IC_NORMALIZATION must be 'max' or 'zero_flux'.")


def squid_envelope(flux_values):
    return np.abs(np.cos(np.pi * flux_values))


def fraunhofer_envelope(flux_values):
    x = np.pi * flux_values
    y = np.ones_like(x)

    mask = np.abs(x) > 1.0e-12
    y[mask] = np.abs(np.sin(x[mask]) / x[mask])

    return y


# ============================================================
# Plot: geometry
# ============================================================

def plot_geometry_preview(params, plot_params):
    ensure_dir(plot_params["PLOT_DIR"])

    lx = lx_total(params)
    wy = params["WY"]

    region_map = np.zeros((wy, lx))

    for x in range(lx):
        reg = region_of_x(x, params)

        if reg == "L":
            val = 0.0
        elif reg == "QSH":
            val = 0.5
        else:
            val = 1.0

        region_map[:, x] = val

    fig, ax = plt.subplots(figsize=plot_params["FIGSIZE_GEOMETRY"])

    ax.imshow(
        region_map,
        origin="lower",
        aspect="auto",
        cmap=plot_params["GEOMETRY_CMAP"],
        interpolation="nearest",
        extent=[0, lx, 0, wy],
    )

    ax.axvline(params["LX_S_LEFT"], color="w", lw=1.4, ls="--")
    ax.axvline(params["LX_S_LEFT"] + params["LX_QSH"], color="w", lw=1.4, ls="--")

    ax.text(
        0.5 * params["LX_S_LEFT"],
        0.5 * wy,
        "S",
        color="w",
        ha="center",
        va="center",
        fontsize=16,
    )

    ax.text(
        params["LX_S_LEFT"] + 0.5 * params["LX_QSH"],
        0.5 * wy,
        "QSH",
        color="w",
        ha="center",
        va="center",
        fontsize=16,
    )

    ax.text(
        params["LX_S_LEFT"] + params["LX_QSH"] + 0.5 * params["LX_S_RIGHT"],
        0.5 * wy,
        "S",
        color="w",
        ha="center",
        va="center",
        fontsize=16,
    )

    beautify_axes(
        ax,
        r"$x$",
        r"$y$",
        plot_params,
    )

    savefig_no_margin(
        fig,
        os.path.join(plot_params["PLOT_DIR"], "geometry_preview_QSH_SNS_flux.png"),
        dpi=plot_params["SAVE_DPI"],
    )

    plt.close(fig)


# ============================================================
# Plot: Ic versus flux
# ============================================================

def plot_interference_pattern(data, params, plot_params):
    ensure_dir(plot_params["PLOT_DIR"])

    flux = data["flux_values"]
    Ic = data["Ic_abs"]

    Ic_norm = normalize_ic(flux, Ic, plot_params)

    fig, ax = plt.subplots(figsize=plot_params["FIGSIZE_INTERFERENCE"])

    ax.plot(
        flux,
        Ic_norm,
        color=plot_params["COLOR_IC"],
        lw=plot_params["LW_MAIN"],
        label=plot_params["LABEL_NUMERICAL"],
    )

    if plot_params["SHOW_SQUID_REFERENCE"]:
        ax.plot(
            flux,
            squid_envelope(flux),
            color=plot_params["COLOR_SQUID"],
            lw=plot_params["LW_REFERENCE"],
            ls="--",
            label=r"$|\cos(\pi\Phi/\Phi_0)|$",
        )

    if plot_params["SHOW_FRAUNHOFER_REFERENCE"]:
        ax.plot(
            flux,
            fraunhofer_envelope(flux),
            color=plot_params["COLOR_FRAUNHOFER"],
            lw=plot_params["LW_REFERENCE"],
            ls=":",
            label=r"$|\sin(\pi\Phi/\Phi_0)/(\pi\Phi/\Phi_0)|$",
        )

    ax.axhline(0.0, color=plot_params["COLOR_GUIDE"], lw=0.8, ls="--", zorder=0)
    ax.axvline(0.0, color=plot_params["COLOR_GUIDE"], lw=0.8, ls="--", zorder=0)

    ax.set_xlim(plot_params["FLUX_XLIM"])
    ax.set_ylim(plot_params["IC_YLIM"])

    ax.set_xticks(plot_params["FLUX_XTICKS"])
    ax.set_yticks(plot_params["IC_YTICKS"])

    beautify_axes(
        ax,
        r"$\Phi/\Phi_0$",
        r"$I_c(\Phi)/I_c(0)$",
        plot_params,
    )

    if plot_params["SHOW_LEGEND"]:
        leg = ax.legend(
            loc=plot_params["LEGEND_LOC"],
            frameon=True,
            fancybox=True,
            framealpha=0.92,
            fontsize=plot_params["LEGEND_SIZE"],
            edgecolor="0.75",
        )
        leg.get_frame().set_linewidth(0.8)

    savefig_no_margin(
        fig,
        os.path.join(plot_params["PLOT_DIR"], "QSH_SNS_SQUID_interference.png"),
        dpi=plot_params["SAVE_DPI"],
    )

    plt.close(fig)


# ============================================================
# Plot: current density map I(phi, flux)
# ============================================================

def plot_current_map(data, params, plot_params):
    ensure_dir(plot_params["PLOT_DIR"])

    flux_values = data["flux_values"]
    phi_values = data["phi_values"]
    I_map = data["I_map"]

    I_scale = np.max(np.abs(I_map)) + 1.0e-30

    fig, ax = plt.subplots(figsize=plot_params["FIGSIZE_MAP"])

    dphi = phi_values[1] - phi_values[0]

    im = ax.imshow(
        I_map / I_scale,
        origin="lower",
        aspect="auto",
        extent=[
            phi_values[0] / np.pi,
            (phi_values[-1] + dphi) / np.pi,
            flux_values[0],
            flux_values[-1],
        ],
        cmap=plot_params["CURRENT_MAP_CMAP"],
        vmin=plot_params["CURRENT_MAP_VMIN"],
        vmax=plot_params["CURRENT_MAP_VMAX"],
        interpolation=plot_params["MAP_INTERPOLATION"],
    )

    cbar = fig.colorbar(im, ax=ax, pad=plot_params["CBAR_PAD"])
    cbar.set_label(plot_params["CBAR_LABEL"], fontsize=plot_params["CBAR_LABEL_SIZE"])
    cbar.ax.tick_params(
        labelsize=plot_params["CBAR_TICK_SIZE"],
        width=1.0,
        length=4,
    )

    beautify_axes(
        ax,
        r"$\varphi/\pi$",
        r"$\Phi/\Phi_0$",
        plot_params,
    )

    savefig_no_margin(
        fig,
        os.path.join(plot_params["PLOT_DIR"], "QSH_SNS_current_phase_flux_map.png"),
        dpi=plot_params["SAVE_DPI"],
    )

    plt.close(fig)


# ============================================================
# Plot: selected CPR curves
# ============================================================

def plot_selected_cpr(data, params, plot_params):
    ensure_dir(plot_params["PLOT_DIR"])

    flux_values = data["flux_values"]
    phi_values = data["phi_values"]
    I_map = data["I_map"]

    selected_fluxes = plot_params["SELECTED_FLUXES"]
    colors = plot_params["SELECTED_CPR_COLORS"]

    fig, ax = plt.subplots(figsize=plot_params["FIGSIZE_CPR"])

    if plot_params["CPR_NORMALIZATION"] == "global":
        I_scale = np.max(np.abs(I_map)) + 1.0e-30
    elif plot_params["CPR_NORMALIZATION"] == "each_curve":
        I_scale = None
    else:
        raise ValueError("CPR_NORMALIZATION must be 'global' or 'each_curve'.")

    for i, flux_target in enumerate(selected_fluxes):
        color = colors[i % len(colors)]

        idx = nearest_index(flux_values, flux_target)
        flux_actual = flux_values[idx]

        y = I_map[idx].copy()

        if plot_params["CPR_NORMALIZATION"] == "global":
            y = y / I_scale
        else:
            y = y / (np.max(np.abs(y)) + 1.0e-30)

        ax.plot(
            phi_values / np.pi,
            y,
            color=color,
            lw=plot_params["LW_MAIN"],
            label=rf"$\Phi/\Phi_0={flux_actual:.2f}$",
        )

    ax.axhline(0.0, color=plot_params["COLOR_GUIDE"], lw=0.9, ls="--", zorder=0)

    ax.set_xlim(plot_params["PHI_XLIM"])
    ax.set_xticks(plot_params["PHI_XTICKS"])

    beautify_axes(
        ax,
        r"$\varphi/\pi$",
        r"$I(\varphi,\Phi)/I_{\max}$",
        plot_params,
    )

    if plot_params["SHOW_LEGEND"]:
        leg = ax.legend(
            loc=plot_params["LEGEND_LOC"],
            frameon=True,
            fancybox=True,
            framealpha=0.92,
            fontsize=plot_params["LEGEND_SIZE"],
            edgecolor="0.75",
        )
        leg.get_frame().set_linewidth(0.8)

    savefig_no_margin(
        fig,
        os.path.join(plot_params["PLOT_DIR"], "QSH_SNS_selected_CPR_vs_flux.png"),
        dpi=plot_params["SAVE_DPI"],
    )

    plt.close(fig)


# ============================================================
# Plot: low-energy spectrum at selected flux
# ============================================================

def plot_selected_spectrum(data, params, plot_params):
    ensure_dir(plot_params["PLOT_DIR"])

    flux_values = data["flux_values"]
    phi_values = data["phi_values"]
    low_positive_map = data["low_positive_map"]

    selected_fluxes = plot_params["SELECTED_SPECTRUM_FLUXES"]

    for flux_target in selected_fluxes:
        idx = nearest_index(flux_values, flux_target)
        flux_actual = flux_values[idx]

        low_positive = low_positive_map[idx]

        fig, ax = plt.subplots(figsize=plot_params["FIGSIZE_SPECTRUM"])

        for n in range(low_positive.shape[1]):
            ax.plot(
                phi_values / np.pi,
                low_positive[:, n],
                color=plot_params["COLOR_SPECTRUM"],
                lw=plot_params["LW_SPECTRUM"],
                alpha=plot_params["ALPHA_SPECTRUM"],
            )

        ax.axhline(0.0, color=plot_params["COLOR_GUIDE"], lw=0.9, ls="--", zorder=0)

        ax.set_xlim(plot_params["PHI_XLIM"])
        ax.set_xticks(plot_params["PHI_XTICKS"])

        if plot_params["SPECTRUM_YLIM"] is not None:
            ax.set_ylim(plot_params["SPECTRUM_YLIM"])

        beautify_axes(
            ax,
            r"$\varphi/\pi$",
            r"$E_n(\varphi)$",
            plot_params,
        )

        ax.set_title(
            rf"$\Phi/\Phi_0={flux_actual:.2f}$",
            fontsize=plot_params["TITLE_SIZE"],
        )

        savefig_no_margin(
            fig,
            os.path.join(
                plot_params["PLOT_DIR"],
                f"QSH_SNS_low_energy_spectrum_flux_{flux_actual:.2f}.png",
            ),
            dpi=plot_params["SAVE_DPI"],
        )

        plt.close(fig)


# ============================================================
# Plot all from file
# ============================================================

def plot_all_from_file(data_file, plot_params):
    data, saved_params = load_saved_data(data_file)

    set_plot_style(plot_params)
    ensure_dir(plot_params["PLOT_DIR"])

    print("=" * 80)
    print(f"Loaded data from: {data_file}")
    print(f"Plot directory:   {plot_params['PLOT_DIR']}")
    print("=" * 80)

    if plot_params["PLOT_GEOMETRY"]:
        print("Plotting geometry preview ...")
        plot_geometry_preview(saved_params, plot_params)

    if plot_params["PLOT_INTERFERENCE"]:
        print("Plotting SQUID-like interference pattern ...")
        plot_interference_pattern(data, saved_params, plot_params)

    if plot_params["PLOT_CURRENT_MAP"]:
        print("Plotting current phase-flux map ...")
        plot_current_map(data, saved_params, plot_params)

    if plot_params["PLOT_SELECTED_CPR"]:
        print("Plotting selected CPR curves ...")
        plot_selected_cpr(data, saved_params, plot_params)

    if plot_params["PLOT_SELECTED_SPECTRUM"]:
        print("Plotting selected low-energy spectra ...")
        plot_selected_spectrum(data, saved_params, plot_params)

    print("=" * 80)
    print("Plotting done.")
    print("=" * 80)


# ============================================================
# Main control
# ============================================================

if __name__ == "__main__":

    # ========================================================
    # Run switches
    # ========================================================

    RUN_CALCULATION = True
    RUN_PLOTTING = True

    # RUN_CALCULATION = False, RUN_PLOTTING = True:
    # only redraw figures from saved data.

    # ========================================================
    # Calculation parameters
    # ========================================================

    OUTDIR = "sns_QSH_swave_SQUID_flux"

    DATA_FILE = os.path.join(
        OUTDIR,
        "QSH_SNS_SQUID_flux_data.npz",
    )

    params = dict(
        # ---------------- Output
        OUTDIR=OUTDIR,
        DATA_FILE=DATA_FILE,

        # ---------------- Geometry
        LX_S_LEFT=3,
        LX_QSH=8,
        LX_S_RIGHT=3,
        WY=10,

        # ---------------- s-wave superconducting leads
        T_S=1.0,
        MU_S=0.0,
        DELTA0=0.20,

        # ---------------- S-QSH interface
        T_INTERFACE=0.65,

        # ---------------- QSH/BHZ region
        A_QSH=1.0,
        B_QSH=1.0,
        M_QSH=1.0,
        MU_QSH=0.0,

        # ---------------- Flux scan
        FLUX_MIN=-3.0,
        FLUX_MAX=3.0,
        N_FLUX=41,

        # ---------------- Josephson phase scan
        PHI_MIN=0.0,
        PHI_MAX=2.0 * np.pi,
        N_PHI=61,

        # ---------------- Numerics
        EIG_CUTOFF=1.0e-10,
        N_SPEC_EIGS=24,
        CURRENT_SIGN=1.0,
        SUBTRACT_CURRENT_MEAN=True,
    )

    # ========================================================
    # Plot parameters
    # ========================================================

    plot_params = dict(
        # ---------------- Output
        PLOT_DIR=os.path.join(OUTDIR, "figures"),

        # ---------------- Which plots
        PLOT_GEOMETRY=True,
        PLOT_INTERFERENCE=True,
        PLOT_CURRENT_MAP=True,
        PLOT_SELECTED_CPR=True,
        PLOT_SELECTED_SPECTRUM=True,

        # ---------------- General style
        FONT_FAMILY="Times New Roman",
        MATH_FONTSET="stix",
        FIGURE_DPI=180,
        SAVE_DPI=450,

        AXES_LINEWIDTH=1.2,
        TICK_WIDTH=1.2,
        TICK_LENGTH=5,
        LABEL_SIZE=15,
        TICK_SIZE=12,
        LEGEND_SIZE=10.5,
        TITLE_SIZE=13,

        LW_MAIN=2.0,
        LW_REFERENCE=1.4,
        LW_SPECTRUM=0.85,
        ALPHA_SPECTRUM=0.75,

        SHOW_LEGEND=True,
        LEGEND_LOC="best",

        # ---------------- Figure sizes
        FIGSIZE_GEOMETRY=(6.0, 2.4),
        FIGSIZE_INTERFERENCE=(5.8, 4.0),
        FIGSIZE_MAP=(6.2, 4.2),
        FIGSIZE_CPR=(5.8, 4.0),
        FIGSIZE_SPECTRUM=(5.6, 4.0),

        # ---------------- Colors
        COLOR_IC="#D62728",
        COLOR_SQUID="0.25",
        COLOR_FRAUNHOFER="0.55",
        COLOR_GUIDE="0.70",
        COLOR_SPECTRUM="#1F1F1F",
        LABEL_NUMERICAL="BdG",

        # ---------------- Geometry
        GEOMETRY_CMAP="viridis",

        # ---------------- SQUID interference plot
        IC_NORMALIZATION="zero_flux",   # "zero_flux" or "max"
        SHOW_SQUID_REFERENCE=True,
        SHOW_FRAUNHOFER_REFERENCE=False,

        FLUX_XLIM=(-3.0, 3.0),
        IC_YLIM=(-0.02, 1.08),
        FLUX_XTICKS=list(np.arange(-3, 4, 1)),
        IC_YTICKS=[0.0, 0.5, 1.0],

        # ---------------- Density map controls
        # User-controllable:
        CURRENT_MAP_CMAP="bwr",
        MAP_INTERPOLATION="spline16",
        CURRENT_MAP_VMIN=-1.0,
        CURRENT_MAP_VMAX=1.0,
        CBAR_PAD=0.02,
        CBAR_LABEL=r"$I/I_{\max}$",
        CBAR_LABEL_SIZE=13,
        CBAR_TICK_SIZE=10,

        # ---------------- CPR plot
        SELECTED_FLUXES=[0.0, 0.25, 0.5, 0.75, 1.0],
        SELECTED_CPR_COLORS=[
            "#D62728",
            "#1F77B4",
            "#2CA02C",
            "#9467BD",
            "#FF7F0E",
        ],
        CPR_NORMALIZATION="global",     # "global" or "each_curve"

        # ---------------- Low-energy spectrum
        SELECTED_SPECTRUM_FLUXES=[0.0, 0.5, 1.0],
        SPECTRUM_YLIM=(0.0, 0.35),

        # ---------------- Common axes
        PHI_XLIM=(0.0, 2.0),
        PHI_XTICKS=[0.0, 0.5, 1.0, 1.5, 2.0],
    )

    # ========================================================
    # Print summary
    # ========================================================

    print("=" * 80)
    print("Finite-width S-QSH-S Josephson junction with orbital flux")
    print("Expected pattern: SQUID-like edge-state interference")
    print("=" * 80)
    print(f"RUN_CALCULATION = {RUN_CALCULATION}")
    print(f"RUN_PLOTTING    = {RUN_PLOTTING}")
    print(f"OUTDIR          = {OUTDIR}")
    print(f"DATA_FILE       = {DATA_FILE}")
    print(f"HAS_SCIPY_LINALG = {HAS_SCIPY_LINALG}")
    print("-" * 80)
    print(f"LX_S_LEFT       = {params['LX_S_LEFT']}")
    print(f"LX_QSH          = {params['LX_QSH']}")
    print(f"LX_S_RIGHT      = {params['LX_S_RIGHT']}")
    print(f"WY              = {params['WY']}")
    print(f"NSITES          = {nsites(params)}")
    print(f"Electron dim    = {electron_dim(params)}")
    print(f"BdG dim         = {bdg_dim(params)}")
    print("-" * 80)
    print(f"QSH parameters  = A={params['A_QSH']}, B={params['B_QSH']}, M={params['M_QSH']}")
    print(f"MU_QSH          = {params['MU_QSH']}")
    print(f"DELTA0          = {params['DELTA0']}")
    print(f"T_INTERFACE     = {params['T_INTERFACE']}")
    print("-" * 80)
    print(
        f"Flux range      = "
        f"[{params['FLUX_MIN']}, {params['FLUX_MAX']}], "
        f"N_FLUX = {params['N_FLUX']}"
    )
    print(f"N_PHI           = {params['N_PHI']}")
    print("=" * 80)

    # ========================================================
    # Run
    # ========================================================

    if RUN_CALCULATION:
        calculate_and_save_data(params)

    if RUN_PLOTTING:
        plot_all_from_file(DATA_FILE, plot_params)

    print("=" * 80)
    print("All tasks finished.")
    print("=" * 80)

Finite-width S-QSH-S Josephson junction with orbital flux
Expected pattern: SQUID-like edge-state interference
RUN_CALCULATION = True
RUN_PLOTTING    = True
OUTDIR          = sns_QSH_swave_SQUID_flux
DATA_FILE       = sns_QSH_swave_SQUID_flux/QSH_SNS_SQUID_flux_data.npz
HAS_SCIPY_LINALG = True
--------------------------------------------------------------------------------
LX_S_LEFT       = 3
LX_QSH          = 8
LX_S_RIGHT      = 3
WY              = 10
NSITES          = 140
Electron dim    = 560
BdG dim         = 1120
--------------------------------------------------------------------------------
QSH parameters  = A=1.0, B=1.0, M=1.0
MU_QSH          = 0.0
DELTA0          = 0.2
T_INTERFACE     = 0.65
--------------------------------------------------------------------------------
Flux range      = [-3.0, 3.0], N_FLUX = 41
N_PHI           = 61
Flux    1/41: Phi/Phi0 = -3.000000
Flux    2/41: Phi/Phi0 = -2.850000
Flux    3/41: Phi/Phi0 = -2.700000
Flux    4/41: Phi/Phi0 = -2.550000
Flux 